# Group Time

Group-level response-timing analysis.

**Reads:** `data/group_results/ (10-participant summaries)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load duration data
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# build session stats
rows = []
for _, row in duration_df.iterrows():
    for s_idx, s_name in enumerate(['Session 01', 'Session 02', 'Session 03'], 1):
        rows.append({
            'Participant': row['Participant'],
            'Session': f'Session {s_idx:02d}',
            'Count': '-',
            'Duration STD (s)': round(row[s_name], 2)
        })

session_stats = pd.DataFrame(rows)

print("Duration STD During 3 Sessions for Each Participant")
print(session_stats.to_string(index=False))

# convert to numeric
session_stats['Duration STD (s)'] = pd.to_numeric(session_stats['Duration STD (s)'])

# visualize results bar
plt.figure(figsize=(15, 8))
sns.barplot(x='Session', y='Duration STD (s)', hue='Participant', data=session_stats, errorbar=None)
plt.title('Duration STD for Each Session by Participant')
plt.ylabel('Duration STD (s)')
plt.xlabel('Session')
plt.legend(title='Participant', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine(trim=True)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
plt.close()

# plot per participant
sns.catplot(x='Session', y='Duration STD (s)', col='Participant', data=session_stats, kind='bar', col_wrap=4, height=4, aspect=1)
plt.subplots_adjust(top=0.9)
plt.suptitle('Duration STD for Each Session by Participant')
plt.show()
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

OUTPUT = '../data/group_results'

# load duration data
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# pivot by session
mean_durations = duration_df.set_index('Participant')
mean_durations.columns = ['Session 01', 'Session 02', 'Session 03']

# standardize data
scaler = StandardScaler()
mean_durations_scaled = scaler.fit_transform(mean_durations)

# kmeans clustering
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
clusters = kmeans.fit_predict(mean_durations_scaled)

# add cluster labels
mean_durations['Cluster'] = clusters

# build session stats
rows = []
for _, row in duration_df.iterrows():
    for s_idx, s_name in enumerate(['Session 01', 'Session 02', 'Session 03'], 1):
        rows.append({
            'Participant': row['Participant'],
            'Session': f'Session {s_idx:02d}',
            'Count': '-',
            'Duration STD (s)': round(row[s_name], 2)
        })

session_stats = pd.DataFrame(rows)

print("Duration STD During 3 Sessions for Each Participant")
print(session_stats.to_string(index=False))

# convert to numeric
session_stats['Duration STD (s)'] = pd.to_numeric(session_stats['Duration STD (s)'])

# plot clusters heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(mean_durations.drop('Cluster', axis=1).assign(Cluster=clusters).sort_values('Cluster').drop('Cluster', axis=1), annot=True, cmap='coolwarm', center=0)
plt.title('Participant Clusters Based on Answer Duration Patterns')
plt.show()
plt.close()

# cluster assignments
print("Cluster Assignments:")
print(mean_durations[['Cluster']].reset_index())

# visualize clusters
plt.figure(figsize=(15, 8))
for cluster in range(kmeans.n_clusters):
    participants_in_cluster = mean_durations[mean_durations['Cluster'] == cluster].index
    cluster_data = mean_durations.loc[participants_in_cluster].drop('Cluster', axis=1).T
    for i, participant in enumerate(cluster_data.columns):
        label = f'Cluster {cluster+1}' if i == 0 else '_nolegend_'
        sns.lineplot(data=cluster_data[participant], label=label, marker='o')

plt.title('Duration STD Across Sessions by Clusters')
plt.ylabel('Duration STD (s)')
plt.xlabel('Session')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
plt.close()

# visualize results bar
plt.figure(figsize=(15, 8))
sns.barplot(x='Session', y='Duration STD (s)', hue='Participant', data=session_stats, errorbar=None)
plt.title('Duration STD for Each Session by Participant')
plt.ylabel('Duration STD (s)')
plt.xlabel('Session')
plt.legend(title='Participant', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine(trim=True)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
plt.close()

# plot per participant
sns.catplot(x='Session', y='Duration STD (s)', col='Participant', data=session_stats, kind='bar', col_wrap=4, height=4, aspect=1)
plt.subplots_adjust(top=0.9)
plt.suptitle('Duration STD for Each Session by Participant')
plt.show()
plt.close()